In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR

BASE_REGISTRY: Path =  REGISTRIES_DIR / "base_output_registry.csv"
EMBEDDING_REGISTRY: Path = SUBREGISTRIES_DIR / "embedding_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"

df_base_registry = pd.read_csv(BASE_REGISTRY)
df_embeddding_registry = pd.read_csv(EMBEDDING_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)


embeddings = []
dois = []
file_paths = []

merge_base_embedding = df_embeddding_registry.merge(df_base_registry[["output_sha", "doc_doi", "output_type", "dependencies"]], on="output_sha", how="left")

merge_base_emb_extraction = merge_base_embedding.merge(
    df_extraction_registry,
    left_on="dependencies",
    right_on="output_sha",
    how="left",
    suffixes=("_embed", "_extract"))



embedding_rows = merge_base_emb_extraction.loc[merge_base_embedding["output_type"]=="embedding"]

for _, row in tqdm(embedding_rows.iterrows(), desc="Loading embeddings", unit="row(s)"):

    embedding_path = Path(row["file_path_embed"])

    cas_section_path = Path(row["file_path_extract"])

    if embedding_path.exists():

        emb = np.load(embedding_path)

        embeddings.append(emb)

        dois.append(row["doc_doi"])

        file_paths.append(cas_section_path)
    
embeddings = np.array(embeddings)

print(embeddings.shape)

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, metric="cosine")

embeddings_2d = reducer.fit_transform(embeddings)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=30, metric="euclidean")

labels = clusterer.fit_predict(embeddings_2d)

print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Noise points: {(labels == -1).sum()}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels, cmap="tab20", alpha=0.5, s=5)

plt.colorbar(scatter)

plt.title("CAS embedding clusters")

plt.show()

In [ ]:
df_results = pd.DataFrame(
    {"doc_doi":dois,
     "cluster": labels,
     "file_path": file_paths}
)

cluster_sizes = df_results["cluster"].value_counts().to_dict()

for cluster_id in sorted(set(labels)):

    if cluster_id == -1:

        continue

    print(f"\n=== Cluster {cluster_id} | size={cluster_sizes.get(cluster_id, 0)} ===")

    sample = df_results[df_results["cluster"]==cluster_id].sample(min(5, len(df_results[df_results["cluster"] == cluster_id])))

    for _, row in sample.iterrows():
        print(Path(row["file_path"]).read_text(encoding="utf-8", errors="replace"))
        print("---")

In [ ]:
df_results["text"] = df_results["file_path"].apply(
    lambda fp: Path(fp).read_text(encoding="utf-8", errors="replace")
)

In [ ]:
import re


def normalize_text(text):
    text = text.lower()
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_results["text_norm"] = df_results["text"].apply(normalize_text)

In [ ]:
df_cluster_1 = df_results.loc[df_results["cluster"]==1]

df_cluster_1["text_norm"].value_counts()

In [ ]:
df_cluster_2 = df_results.loc[df_results["cluster"]==2]
df_cluster_3 = df_results.loc[df_results["cluster"]==3]
df_cluster_4 = df_results.loc[df_results["cluster"]==4]
df_cluster_5 = df_results.loc[df_results["cluster"]==5]
df_cluster_6 = df_results.loc[df_results["cluster"]==6]
df_cluster_7 = df_results.loc[df_results["cluster"]==7]
df_cluster_8 = df_results.loc[df_results["cluster"]==8]
df_cluster_9 = df_results.loc[df_results["cluster"]==9]
df_cluster_10 = df_results.loc[df_results["cluster"]==10]
df_cluster_11 = df_results.loc[df_results["cluster"]==11]


display(df_cluster_2["text_norm"].value_counts())
display(df_cluster_3["text_norm"].value_counts())
display(df_cluster_4["text_norm"].value_counts())
display(df_cluster_5["text_norm"].value_counts())
display(df_cluster_6["text_norm"].value_counts())
display(df_cluster_7["text_norm"].value_counts())
display(df_cluster_8["text_norm"].value_counts())
display(df_cluster_9["text_norm"].value_counts())
display(df_cluster_10["text_norm"].value_counts())
display(df_cluster_11["text_norm"].value_counts())

In [ ]:
df_cluster2_doi = df_results.loc[df_results["cluster"]==2, ["doc_doi", "text"]]
df_cluster11_doi = df_results.loc[df_results["cluster"]==11, ["doc_doi", "text"]]

In [ ]:
display(df_cluster2_doi)

In [ ]:
display(df_cluster11_doi)

In [ ]:
df_cluster_0 = df_results.loc[df_results["cluster"]==0]

df_cluster_0["text_norm"].value_counts()

In [ ]:
for s in df_cluster_0["text"].unique():
    print(repr(s))

In [ ]:
df_cluster_11["text"].value_counts()

In [ ]:
df_results.groupby("cluster")["text"].nunique()